# 08 Test Inference and Adapter Specialization

This notebook sends representative requests to each adapter and compares outputs.

```mermaid
flowchart TB
    A[Prompt set] --> B[finance adapter]
    A --> C[legal adapter]
    A --> D[healthcare adapter]
    B --> E[Compare tone and terminology]
    C --> E
    D --> E
```

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Example prompts and expected behavior

- Finance should mention financial risk, margin, cash, revenue, or accounting context.
- Legal should mention contracts, clauses, liability, or counsel.
- Healthcare should mention coverage, care plans, discharge, or patient administration.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{settings_cfg.vllm_base_url}/v1", api_key=settings_cfg.vllm_api_key)
prompts = {
    "finance": "Explain revenue concentration risk in two sentences.",
    "legal": "Explain why a limitation of liability clause matters.",
    "healthcare": "Explain what prior authorization means.",
}

outputs = {}
for adapter, prompt in prompts.items():
    response = client.chat.completions.create(
        model=adapter,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=180,
    )
    outputs[adapter] = response.choices[0].message.content
    print(f"\n## {adapter}\nPrompt: {prompt}\nOutput: {outputs[adapter]}")

## Lightweight evaluation

Expected output: keyword scores logged to MLflow.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(PROJECT_ROOT / "evaluation" / "evaluate.py")], check=True)